# **RAG INTERNSHIP PROJECT**

**Objective:**

The objective of this project is to develop an AI-powered customer support assistant that can answer user queries based on information available in a router user manual. The system uses a Retrieval-Augmented Generation (RAG) approach where the manual is processed, split into smaller text chunks and converted into embeddings using models from Sentence Transformers. These embeddings are stored in a vector database using ChromaDB to enable semantic search. When a user asks a question, the system retrieves the most relevant information from the manual and sends it to a language model via Groq to generate a clear and accurate response. If the system cannot find relevant information in the knowledge base, the query is escalated to human support, implementing a Human-in-the-Loop (HITL) mechanism to ensure reliable assistance.

**Install Required Libraries**

In [6]:
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-c

In [7]:
import langchain
import langchain_community

print("LangChain installed successfully")

LangChain installed successfully


**Load the PDF**

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/router_manual.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 110


**Split the Document (Chunking)**

In [10]:
!pip install -q langchain-text-splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 343


**Create Embeddings**

In [13]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded")

**Store in Vector Database (ChromaDB)**

In [15]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    chunks,
    embeddings
)

print("Vector database created")

Vector database created


**Retrieve Relevant Information**

In [16]:
query = "How do I reset the router?"

results = db.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("-----")

To reset the router, locate the reset button (hole) on the rear panel of the unit. With the router powered on, use a paperclip 
to hold the button down for 10 seconds. Release the button and the router will go through its reboot process. Wait about 30 
seconds to access the router. The default IP address is 192.168.0.1. When logging in, leave the password box empty.
Downloaded from www.Manualslib.com manuals search engine
-----
• If you still cannot access the configuration, unplug the power to the router for 10 seconds and plug back in. Wait about 30 
seconds and try accessing the configuration. If you have multiple computers, try connecting using a different computer.
2. What can I do if I forgot my password?
If you forgot your password, you must reset your router. This process will change all your settings back to the factory defaults.
-----
settings that have not been backed up will be lost, including any 
rules that you have created.
This option will reboot the router.
Save Settin

**Install the LLM integration**

In [17]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


In [19]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq chromadb sentence-transformers pypdf

In [23]:
!pip install -q groq

In [24]:
from groq import Groq

client = Groq(api_key="your api key")

**Use an LLM to generate answers**

In [25]:
query = "How do I reset the router?"

docs = db.similarity_search(query, k=3)

context = "\n".join([d.page_content for d in docs])

print(context)

To reset the router, locate the reset button (hole) on the rear panel of the unit. With the router powered on, use a paperclip 
to hold the button down for 10 seconds. Release the button and the router will go through its reboot process. Wait about 30 
seconds to access the router. The default IP address is 192.168.0.1. When logging in, leave the password box empty.
Downloaded from www.Manualslib.com manuals search engine
• If you still cannot access the configuration, unplug the power to the router for 10 seconds and plug back in. Wait about 30 
seconds and try accessing the configuration. If you have multiple computers, try connecting using a different computer.
2. What can I do if I forgot my password?
If you forgot your password, you must reset your router. This process will change all your settings back to the factory defaults.
settings that have not been backed up will be lost, including any 
rules that you have created.
This option will reboot the router.
Save Settings To 
Local

**Final Generation Step**

In [27]:
query = "How do I reset the router?"

docs = db.similarity_search(query, k=3)

context = "\n".join([d.page_content for d in docs])

prompt = f"""
You are a customer support assistant.

Answer the user's question clearly using ONLY the information from the context.

Context:
{context}

Question:
{query}

Give a short helpful answer.
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

To reset the router, locate the reset button on the rear panel, and use a paperclip to hold it down for 10 seconds while the router is powered on. The router will then go through its reboot process, and its default settings will be restored.


**Human-in-the-Loop (HITL)**

In [28]:
if len(docs) == 0:
    print("I couldn't find the answer in the knowledge base.")
    human = input("Human support agent response: ")
    print("Support Agent:", human)

**Create a Chat Loop**

In [33]:
while True:

    query = input("\nAsk your question (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    results = db.similarity_search_with_score(query, k=3)

    docs = [doc for doc, score in results]

    # check relevance
    if results[0][1] > 0.6:
        print("This question is outside the knowledge base.")
        human = input("Human support response: ")
        print("Support Agent:", human)
        continue

    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
    You are a customer support assistant.

    Answer using ONLY the context.

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    print("\nAssistant:", response.choices[0].message.content)


Ask your question (type 'exit' to stop): how to reset router

Assistant: To reset the router, locate the reset button on the rear panel of the unit. With the router powered on, use a paperclip to hold the button down for 10 seconds. Release the button and the router will go through its reboot process. Wait about 30 seconds to access the router.

Ask your question (type 'exit' to stop): what is bitcoin
This question is outside the knowledge base.
Human support response: Bitcoin is a digital currency.
Support Agent: Bitcoin is a digital currency.

Ask your question (type 'exit' to stop): exit
Goodbye!
